# Multi-echelon supply chain network optimization — walkthrough

One MILP making two kinds of decision at once:

- **Strategic** — which plants and warehouses to open, paid for once, for the
  whole horizon.
- **Operational** — how much to produce, ship on each leg, and hold as
  inventory, period by period.

Solving them together is the point. Choose the network first and you are
guessing at the operating costs it will imply; choose the flows first and the
network is already fixed. This notebook shows the trade-off being made.

The model follows the standard multi-echelon facility-location structure from
the supply chain literature (Melo, Nickel & Saldanha-da-Gama, *EJOR* 2009) —
a well-established class of model, not an original formulation.

**The sample networks are synthetic**, hand-designed so the answer can be
reasoned about rather than taken on trust.

In [ ]:
from pathlib import Path

from scn_opt.data.loaders import load_system_json
from scn_opt.data.schema import Customer, System
from scn_opt.model.builder import build_from_system
from scn_opt.solve import solve_scn
from scn_opt.viz import (
    plot_cost_breakdown,
    plot_inventory_trajectories,
    plot_network,
)

REPO = Path.cwd().parent
NETWORKS = REPO / "data" / "sample_networks"

system = load_system_json(NETWORKS / "baseline.json")

print(f"{len(system.plants)} candidate plants, {len(system.warehouses)} candidate "
      f"warehouses, {len(system.customers)} customers, {system.n_periods} periods")
print(f"demand by period: {system.demand_by_period}")
print(f"total plant capacity per period: {system.total_plant_capacity:g}")

## Solving

`build_from_system` constructs the Pyomo model and `solve_scn` runs HiGHS,
returning the opened facilities, the flows on every lane, the inventory held,
and a cost breakdown that sums to the objective by construction.

In [ ]:
result = solve_scn(build_from_system(system))

print(result.summary())
print()
print(f"open plants     : {result.open_plants}")
print(f"open warehouses : {result.open_warehouses}")
print()
for component, amount in result.cost_breakdown().items():
    print(f"  {component:<18} {amount:>10,.1f}")
print(f"  {'total':<18} {result.total_cost:>10,.1f}")

In [ ]:
plot_network(system, result)

## What is the strategic decision actually worth?

`force_open_all` pins every candidate facility open while still charging its
fixed cost, which gives a like-for-like counterfactual: the same demand served
by the whole candidate network.

Watch which way each cost component moves. The saving does **not** come from
doing less of everything — closing facilities means longer hauls, so shipping
goes *up*. The strategic decision is a trade, and this is the shape of it.

In [ ]:
everything = solve_scn(build_from_system(system, force_open_all=True))

chosen = result.cost_breakdown()
opened_all = everything.cost_breakdown()

print(f"{'component':<18}{'optimized':>12}{'all open':>12}{'change':>12}")
for component in chosen:
    delta = chosen[component] - opened_all[component]
    print(f"{component:<18}{chosen[component]:>12,.1f}{opened_all[component]:>12,.1f}"
          f"{delta:>+12,.1f}")
print(f"{'TOTAL':<18}{result.total_cost:>12,.1f}{everything.total_cost:>12,.1f}"
      f"{result.total_cost - everything.total_cost:>+12,.1f}")

In [ ]:
plot_cost_breakdown(system, result, baseline=everything)

## Fixed against variable, as volume grows

The `tradeoff` network reduces the decision to its essentials: two warehouses,
identical but for how they split their cost between opening and shipping.

    W_cheap_fixed:     100 to open, 5 per unit
    W_cheap_shipping: 1000 to open, 1 per unit

Over four periods those are equal when `100 + 20D = 1000 + 4D`, i.e. at
**D = 56.25** units per period. Below it the cheap-to-open site wins; above it
the cheap-to-ship one does. Nothing tells the model this — it falls out of the
arithmetic.

In [ ]:
tradeoff = load_system_json(NETWORKS / "tradeoff.json")


def at_demand(units_per_period):
    """Same network, different demand level."""
    variant = System(
        plants=tradeoff.plants,
        warehouses=tradeoff.warehouses,
        customers=[Customer(name="C1", demand=[units_per_period] * tradeoff.n_periods)],
        cost_plant_to_warehouse=tradeoff.cost_plant_to_warehouse,
        cost_warehouse_to_customer=tradeoff.cost_warehouse_to_customer,
    )
    return solve_scn(build_from_system(variant))


print(f"{'demand/period':>14}  {'total cost':>11}  warehouse opened")
for units in [20, 40, 55, 56, 57, 70, 100]:
    solved = at_demand(float(units))
    marker = "  <-- flips here" if units == 57 else ""
    print(f"{units:>14}  {solved.total_cost:>11,.1f}  "
          f"{solved.open_warehouses[0]}{marker}")

## Why the model spans periods

The `spike` network has demand of 20 / 150 / 20 / 20 against a plant that can
make only 100 in a period. Period 2 cannot be produced in period 2 — the only
way through is to build early and hold.

This is also why the aggregate feasibility check in the schema is *cumulative*
rather than per period. Production can be banked forward; it cannot be borrowed
from the future.

In [ ]:
spike = load_system_json(NETWORKS / "spike.json")
spike_result = solve_scn(build_from_system(spike))

print(f"demand by period    : {spike.demand_by_period}")
print(f"plant capacity/period: {spike.total_plant_capacity:g}")
print()
for t in spike.periods:
    produced = sum(
        q for (_, _, tt), q in spike_result.flows_plant_to_warehouse.items() if tt == t
    )
    shipped = sum(
        q for (_, _, tt), q in spike_result.flows_warehouse_to_customer.items() if tt == t
    )
    held = spike_result.inventory[("W1", t)]
    print(f"period {t}: produced {produced:>6.1f}   shipped out {shipped:>6.1f}   "
          f"held {held:>6.1f}")

In [ ]:
plot_inventory_trajectories(spike, spike_result)

## A word on safety stock

`safety_stock` is a **flat policy input**, not a service level derived from
demand variability — there is no variability in this deterministic model to
derive one from.

It also behaves in a way worth seeing before it surprises you: the constraint
requires the floor to hold in *every* period, so the buffer is produced once
and then never drawn down. It is not a reserve the plan dips into during a
peak; it is stock that sits there being charged holding cost.

In [ ]:
for warehouse in system.warehouses:
    if warehouse.name not in result.open_warehouses:
        continue
    levels = [result.inventory[(warehouse.name, t)] for t in system.periods]
    print(f"{warehouse.name}: floor {warehouse.safety_stock:g}, "
          f"held {[round(v, 1) for v in levels]}")

total_buffer = sum(
    w.safety_stock for w in system.warehouses if w.name in result.open_warehouses
)
print(f"\ntotal demand over the horizon : {system.total_demand:g}")
print(f"units produced                : "
      f"{sum(result.flows_plant_to_warehouse.values()):g}")
print(f"difference (the buffer, never sold) : {total_buffer:g}")

## Scope and caveats

- **Single product.** No multi-commodity flows.
- **Facilities open for the whole horizon.** No phased opening or closing,
  which would need time-indexed binaries.
- **Demand met exactly.** No backorders, stockouts, or lost sales — an
  unservable instance is infeasible rather than expensive.
- **Safety stock is a policy input**, not statistically derived.
- **No lead times.** Everything shipped arrives in the period it leaves.
- **Constant unit costs.** No economies of scale or quantity discounts.

Full list with rationale in `docs/formulation.md`.